# Selecting Behaviors with Finite-State Machines


### Finite-State Machine

A finite-state machine (FSM) consists of a **finite set of states $s_N$** (where $N$ is the number of states) and a set of **transitions between pairs of states $s_i$ , $s_j$**. The FSM has one (and only one) of its states active at any given moment. Transitions between states can be caused by inputs from sensors or internal events (like a timer or internal counter, for example). An **action** associated to a state is taken when the machine is at that corresponding state. 

_Note 1: Actions can also be taken during transitions, but this is not covered here._

The image below shows the diagram of a FSM with 4 states: S1, S2, S3, S4. The _transitions_ between states are indicated by the black arrows that also show the _event_ that causes it. The text in red shows the _action_ associated to each of the states. 

![Finite-state machine diagram to solve a maze](images/maze-solver_state_machine.png)

### Selecting Behaviors to Complete a Mission

The Finite-State Machine diagram shown above represents the mission control for a mobile robot to solve a maze. A behavior is associated to each of the states, which are selected based on the indicated transitions. Here's a summary of its operation:

* The initial state is S1. 
* While S1 is active, the behavior "follow-wall" will be executed until one of the following events happen:
    * Front obstacle is close $\rightarrow$ transition to state S2
    * Left wall not detected $\rightarrow$ transition to state S3
    * End line detected $\rightarrow$ transition to state S4
* When S2 is active, the behavior "Turn $-90^o$" is activated. The robot will be on this state until the turn ends, then the FSM makes a transition back to S1.
* When S3 is active, the behavior "Curve $+90^o$" is activated. The robot will be on this state until the curve ends, then the FSM makes a transition back to S1.
* When S4 is active, the bevaior "Stop" is activated. There is no transition from S4 to any other state, so this is the final state of the robot.

_Note 2_: The ececution of the FSM described above ends in the final state S4. However, FSMs do not need to have a final state, which means that some FSMs can operate indefinately. 

It is important to realize that events are only considered when the active state has transitions associated to it. For instance, in the above FSM, if an obstacle is detected in front of the robot while the active state is S3, the robot will continue to make the curve until it is completed. If you want the robot to react to a front obstacle while in S3, then you must include another transition associated to this event to change the active state from S3 to another one.

### Python implementation

A Finite-State Machine can be implemented using a sequence of `if` commands that are executed at every loop of the _see-think-act_ cycle. In the code below, `current_state` is a variable that contains the name of the active state (S1, S2, S3 or S4). We assume that the function `follow_wall_to_left(kd, kd2, psValues, d_desired)` generates desired values of linear and angular speeds for the robot to follow a wall located at its left side. The actions related to S2, S3 and S4 are generated by fixed values of desired linear $u_d$ and angular $\omega_d$ speeds.    

```python
# Actions:
if current_state == 'S1':
    u_d, w_d = follow_wall_to_left(kd, kd2, psValues, d_desired)

if current_state == 'S2':
    u_d = 0.0   # No linear speed
    w_d = -0.5  # Rotate right

if current_state == 'S3':
    u_d = 0.03  # Some linear speed
    w_d = 0.6   # Rotate left

if current_state == 'S4':
    u_d = 0.0   # stop
    w_d = 0.0   # stop
```

The snippet above only selects the behavior based on the active state. After that, the code must check if the `current_state` needs to change based on events. For that, it will check for the events associated to each of the states. Note that there is no transition from S4 because that's the final state. 

```python
# Transitions:
if current_state == 'S1':
    if psValues[5] < 80 and psValues[6] < 80:   # Left wall is not detected
        current_state = 'S3'
        counter = 20    
    if psValues[0] > 150 or psValues[7] > 170:  # Front obstacle detected
        current_state = 'S2'
        counter = 0    
    if end_line:
        current_state = 'S4'

if current_state == 'S3':
    if counter >= COUNTER_MAX:
        current_state = 'S1'  

if current_state == 'S2':
    if counter >= COUNTER_MAX:
        current_state = 'S1'

counter += 1    # increment counter
```

A counter is used to count the number of cycles for states S2 and S3 to remain active. Assuming that the _see-think-act_ cycle is executed at a fixed frequency, counting cycles is equivalent to count time. 

The execution of the FSM explained above is illustrated by the animation below. Try to identify the which state is active at each moment, and what events cause transitions between states.

![e-puck robot following a maze with a state machine](images/e-puck_maze_following.gif)

### Conclusion

After completing this notebook, you should understand Finite-State Machines and how to implement them to select behaviors to control robots.

##### Back to the [main page](README.md).